In [19]:
#%load_ext cudf.pandas
#%load_ext cuml.accel

import cudf
import pandas as pd
import cupy as cp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import re
from collections import Counter

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE

device(type='cuda')

In [2]:
df_train = pd.read_csv('datasets/Quora Questions Pair Dataset/02_train_preprocessed.csv')
print(type(df_train))
df_train.sample(3)

<class 'pandas.core.frame.DataFrame'>


,question1,question2,is_duplicate
369060,how is sulphur soluble in water,why is chlorine soluble in water,0
202494,why is saltwater taffy candy imported in switz...,why is saltwater taffy candy imported in austria,1
42974,what is the best way to learn english,how can learn english,1


In [3]:
df_train['is_duplicate'].to_csv('datasets/Quora Questions Pair Dataset/03_y.csv', index=False)

In [4]:
def tokenize(text):
    return text.split()

In [5]:
all_tokens = []
for col in ['question1', 'question2']:
    for row in df_train[col]:
        all_tokens.extend(tokenize(row))
print(f"Corpus size: {len(all_tokens)}")

Corpus size: 9016420


In [6]:
#get all the words with the number of times they repeat
counter = Counter(all_tokens)

first_five=0
for w,c in counter.items():
    if first_five >= 5:
        break
    print(f"{w}: {c}")
    first_five += 1

what: 324583
is: 303747
the: 381218
step: 758
by: 18236


In [7]:
# build vocab of all the words with 5 or more occurences
MIN_COUNT = 4
vocab = ['<pad>','<unk>'] + [w for w,c in counter.items() if c >= MIN_COUNT]
vocab[:5]

['<pad>', '<unk>', 'what', 'is', 'the']

In [8]:
# generate index of all the words
word2idx = {w:i for i,w in enumerate(vocab)}

first_five=0
for w,c in word2idx.items():
    if first_five >= 5:
        break
    print(f"{w}: {c}")
    first_five += 1

<pad>: 0
<unk>: 1
what: 2
is: 3
the: 4


In [9]:
VOCAB_SIZE = len(vocab)
VOCAB_SIZE

28549

In [10]:
# tokenize a text --> consider upto first 25 words only
# if there are <25 words, append zeros at the end to make 25 words exactly
def encode(sent, max_words=25):
    # in word2idx.get(w,1) --> if w doesn't exist in vocab, return index 1 <'unk'>
    ids = [word2idx.get(w,1) for w in tokenize(sent)][:max_words]
    # append 0 at the end, 0=<'pad'> to make the length 25 exactly
    ids += [0] * (max_words - len(ids))
    return ids

In [11]:
MAX_WORDS = 30
df_train['ids1'] = df_train['question1'].apply(lambda s: encode(s, MAX_WORDS))
df_train['ids2'] = df_train['question2'].apply(lambda s: encode(s, MAX_WORDS))
df_train.sample(5)

,question1,question2,is_duplicate,ids1,ids2
67197,what are some really unique names,what are some unique names,1,"[2, 96, 159, 510, 2819, 2860, 0, 0, 0, 0, 0, 0...","[2, 96, 159, 2819, 2860, 0, 0, 0, 0, 0, 0, 0, ..."
35572,which credit card in india gives the most rewards,which credit card in india offers the most cas...,0,"[39, 2103, 104, 10, 13, 3877, 4, 192, 11275, 0...","[39, 2103, 104, 10, 13, 5532, 4, 192, 597, 244..."
31163,how do i prepare for bits in patna what are an...,where can i get good questions to prepare for ...,0,"[21, 68, 18, 177, 114, 3037, 10, 12199, 2, 96,...","[358, 22, 18, 311, 65, 133, 8, 177, 114, 477, ..."
348504,will the new sat be harder than the current sat,how do i compare the new sat with the old sat,0,"[214, 4, 263, 9372, 64, 4252, 411, 4, 115, 937...","[21, 68, 18, 107, 4, 263, 9372, 125, 4, 220, 9..."
358534,what should i do to give my girlfriend orgasm ...,how do i make my girlfriend have an orgasm dur...,1,"[2, 61, 18, 68, 8, 194, 25, 229, 4704, 1914, 2...","[21, 68, 18, 86, 25, 229, 308, 120, 4704, 187,..."


In [18]:
np.savetxt('datasets/Quora Questions Pair Dataset/04_q1_ids.csv', 
           np.stack(df_train['ids1'].values), delimiter=",", fmt='%d')
np.savetxt('datasets/Quora Questions Pair Dataset/05_q2_ids.csv', 
           np.stack(df_train['ids1'].values), delimiter=",", fmt='%d')

In [ ]:
np.loadtxt('datasets/Quora Questions Pair Dataset/04_q1_ids.csv', delimiter=",")
np.loadtxt('datasets/Quora Questions Pair Dataset/05_q2_ids.csv', delimiter=",")

In [14]:
# Skip-gram Word2Vec with negative sampling (using custom nn.Module on GPU)

WINDOW = 4
EMBED_DIM = 128
NEG_SAMPLES = 5

In [15]:
# for each row, a pair is (curr_word, +-WINDOW word)
def build_skipgram_pairs(token_list, window=WINDOW):
    pairs = []
    for tokens in token_list:
        ids = [word2idx.get(w,1) for w in tokens]
        n = len(ids)
        for i, center in enumerate(ids):
            lo = max(0, i - window)
            hi = min(n, i + window + 1)
            for j in range(lo, hi):
                if j != i:
                    pairs.append((center, ids[j]))
    return pairs

In [16]:
sentence_tokens = [tokenize(s) for s in pd.concat([df_train['question1'],df_train['question2']])]
pairs = build_skipgram_pairs(sentence_tokens)
pairs[:5]

[(2, 3), (2, 4), (2, 5), (2, 6), (3, 2)]

In [17]:
pairs = np.array(pairs, dtype=np.int64)
pairs[:5]

array([[2, 3],
       [2, 4],
       [2, 5],
       [2, 6],
       [3, 2]])

In [18]:
word_freq = np.zeros(VOCAB_SIZE)
for w,c in counter.items():
    if w in word2idx:
        word_freq[word2idx[w]] = c
word_freq[:5]

array([     0.,      0., 324583., 303747., 381218.])

In [19]:
neg_dist = torch.tensor(
    (word_freq ** 0.75)/(word_freq ** 0.75).sum(), 
    dtype=torch.float, 
    device=DEVICE
)

neg_dist[:5]

tensor([0.0000, 0.0000, 0.0124, 0.0118, 0.0140], device='cuda:0')

In [20]:
# Custom Dataset Class
class SkipGramDataset(Dataset):
    def __init__(self, pairs):
        self.centers = torch.tensor(pairs[:, 0], dtype=torch.long)
        self.contexts = torch.tensor(pairs[:, 1], dtype=torch.long)

    def __len__(self):
        return len(self.centers)

    def __getitem__(self, idx):
        return self.centers[idx], self.contexts[idx]

In [18]:
# Model class
class SkipGramNS(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        self.out_embed = nn.Embedding(vocab_size, embed_dim)
        nn.init.uniform_(self.in_embed.weight -0.5/embed_dim, 0.5/embed_dim)
        nn.init.constant_(self.out_embed.weight, 0)

    def forward(self, center, context, neg_dist, num_neg=NEG_SAMPLES):
        v = self.in_embed(center)
        u_pos = self.out_embed(context)
        pos_score = torch.sum(v * u_pos, dim=1)
        pos_loss = F.logsigmoid(pos_score)
        neg_idx = torch.multinomial(neg_dist, center.size(0) * num_neg, replacement=True)
        neg_idx = neg_idx.view(center.size(0), num_neg)
        u_neg = self.out_embed(neg_idx)
        neg_score = torch.bmm(u_neg, v.unsqueeze(2)).squeeze(2)
        neg_loss = F.logsigmoid(-neg_score).sum(1)
        return -(pos_loss + neg_loss).mean()

In [19]:
model = SkipGramNS(VOCAB_SIZE, EMBED_DIM).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
loader = DataLoader(SkipGramDataset(pairs), batch_size=4096, shuffle=True, num_workers=0)

In [20]:
import tqdm
from tqdm.auto import tqdm
EPOCHS = 10
for epoch in range(EPOCHS):
    total_loss = 0.0
    for centers, contexts in tqdm(loader, desc='loader...'):
        centers, contexts = centers.to(DEVICE), contexts.to(DEVICE)
        loss = model(centers, contexts, neg_dist)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}/{EPOCHS} loss={total_loss/len(loader):.4f}')

embedding_matrix = model.in_embed.weight.detach()

/home/dex/miniconda3/envs/rapids_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
loader...: 100%|████████████████████████████████████████████████████████████████| 13668/13668 [09:04<00:00, 25.12it/s]


Epoch 1/10 loss=2.2231


loader...: 100%|████████████████████████████████████████████████████████████████| 13668/13668 [09:15<00:00, 24.61it/s]


Epoch 2/10 loss=2.1197


loader...: 100%|████████████████████████████████████████████████████████████████| 13668/13668 [09:09<00:00, 24.88it/s]


...

Epoch 8/10 loss=2.0629


loader...: 100%|████████████████████████████████████████████████████████████████| 13668/13668 [08:58<00:00, 25.39it/s]


Epoch 9/10 loss=2.0610


loader...: 100%|████████████████████████████████████████████████████████████████| 13668/13668 [09:10<00:00, 24.81it/s]

Epoch 10/10 loss=2.0596


In [22]:
torch.save(model, 'models/word_pair_emb_model.pth')

embedding_np = embedding_matrix.cpu().numpy()
embedding_df = pd.DataFrame(embedding_np,
                           columns=[f"dim_{i}" for i in range(embedding_np.shape[1])])
embedding_df.insert(0, 'word', vocab)
embedding_df.to_csv('datasets/Quora Questions Pair Dataset/06_word_pair_emb.csv', index=False)

In [21]:
# loading word pair embeddings from saved file
embedding_df = pd.read_csv('datasets/Quora Questions Pair Dataset/06_word_pair_emb.csv')
vocab = embedding_df['word'].tolist()
embedding_np = np.array(embedding_df.iloc[:,1:])
embedding_matrix = torch.tensor(embedding_np)
embedding_matrix.shape, type(embedding_matrix), embedding_matrix

(torch.Size([28549, 128]),
 torch.Tensor,
 tensor([[ 0.3756,  0.2267, -0.0065,  ..., -0.2938,  0.4096,  1.8223],
         [ 0.0036,  0.0667,  0.4862,  ..., -0.1020,  0.0392, -0.2243],
         [ 0.2502,  0.0046,  1.3010,  ...,  0.1616, -0.0993, -0.0726],
         ...,
         [ 0.9871,  0.8178, -0.6882,  ..., -1.6554, -2.7404, -0.2003],
         [ 0.7834,  2.6968,  2.5663,  ..., -1.0214,  2.3385,  0.3786],
         [-1.4707,  0.7946,  0.0672,  ..., -0.9628,  1.7868, -1.3583]],
        dtype=torch.float64))

In [23]:
np.savetxt('datasets/Quora Questions Pair Dataset/07_emb_matrix.csv', 
           embedding_np, delimiter=",")

In [24]:
embedding_np = np.loadtxt('datasets/Quora Questions Pair Dataset/07_emb_matrix.csv', delimiter=",")

## Create Sentence Embeddings using word Embeddings

In [ ]:
### Create Sentence Embeddings
def ids_to_embedding(id_list_batch): #input is np.array of shape batch_size x 30
    # covert np.array to torch.tensor, dimension = (rows, words per row (=30))
    ids_tensor = torch.tensor(id_list_batch, dtype=torch.long, device=DEVICE)
    
    mask = (ids_tensor != 0)    #Convert tensor to True if there is a value, else False
    mask = mask.unsqueeze(-1)   #Increase a dimension at the end
    mask = mask.float()         #Convert True to 1, False to 0
    
    vecs = embedding_matrix[ids_tensor] # rows, words, 128 per word
    vecs = vecs * mask #zero embeddings for padded word
    
    summed = vecs.sum(dim=1)    # add all word vectors for each row (rows,128)
    counts = mask.sum(dim=1).clamp(min=1)    #count all words per row (rows, 1)
    return summed / counts     #return mean of non-zeros only (rows, 128)

In [18]:
# experimental
sr_ids1 = df_train['ids1'][:10]
sr_ids1.shape, type(sr_ids1), sr_ids1

((10,),
 pandas.core.series.Series,
 0    [2, 3, 4, 5, 6, 5, 7, 8, 9, 10, 11, 12, 10, 13...
 1    [2, 3, 4, 14, 15, 16, 17, 18, 19, 20, 0, 0, 0,...
 2    [21, 22, 18, 23, 4, 24, 15, 25, 26, 27, 28, 29...
 3    [32, 33, 18, 34, 35, 36, 21, 22, 18, 37, 38, 0...
 4    [39, 40, 41, 10, 42, 43, 44, 45, 46, 47, 48, 1...
 5    [50, 18, 33, 30, 51, 52, 53, 54, 47, 53, 55, 2...
 6    [61, 18, 62, 63, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...
 7    [21, 22, 18, 64, 30, 65, 66, 0, 0, 0, 0, 0, 0,...
 8    [67, 68, 69, 70, 1, 71, 15, 1, 0, 0, 0, 0, 0, ...
 9    [72, 73, 22, 18, 74, 25, 75, 72, 76, 0, 0, 0, ...
 Name: ids1, dtype: object)

In [19]:
# experimental
arr_of_list_ids1 = sr_ids1.values
arr_of_list_ids1.shape, type(arr_of_list_ids1), arr_of_list_ids1

((10,),
 numpy.ndarray,
 array([list([2, 3, 4, 5, 6, 5, 7, 8, 9, 10, 11, 12, 10, 13, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
        list([2, 3, 4, 14, 15, 16, 17, 18, 19, 20, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
        list([21, 22, 18, 23, 4, 24, 15, 25, 26, 27, 28, 29, 30, 31, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
        list([32, 33, 18, 34, 35, 36, 21, 22, 18, 37, 38, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
        list([39, 40, 41, 10, 42, 43, 44, 45, 46, 47, 48, 18, 49, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
        list([50, 18, 33, 30, 51, 52, 53, 54, 47, 53, 55, 2, 56, 57, 58, 59, 60, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
        list([61, 18, 62, 63, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
        list([21, 22, 18, 64, 30, 65, 66, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
        list([67, 68, 69, 70, 1, 71, 15, 1, 0, 0, 0,

In [20]:
# experimental
arr_ids1 = np.stack(arr_of_list_ids1)
arr_ids1.shape, type(arr_ids1), arr_ids1

((10, 30),
 numpy.ndarray,
 array([[ 2,  3,  4,  5,  6,  5,  7,  8,  9, 10, 11, 12, 10, 13,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 2,  3,  4, 14, 15, 16, 17, 18, 19, 20,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [21, 22, 18, 23,  4, 24, 15, 25, 26, 27, 28, 29, 30, 31,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [32, 33, 18, 34, 35, 36, 21, 22, 18, 37, 38,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [39, 40, 41, 10, 42, 43, 44, 45, 46, 47, 48, 18, 49,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [50, 18, 33, 30, 51, 52, 53, 54, 47, 53, 55,  2, 56, 57, 58, 59,
         60,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [61, 18, 62, 63,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],


In [21]:
# experimental
tensor_ids1 = torch.tensor(arr_ids1, dtype=torch.long, device=DEVICE)
tensor_ids1.shape, type(tensor_ids1), tensor_ids1

(torch.Size([10, 30]),
 torch.Tensor,
 tensor([[ 2,  3,  4,  5,  6,  5,  7,  8,  9, 10, 11, 12, 10, 13,  0,  0,  0,  0,
           0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 2,  3,  4, 14, 15, 16, 17, 18, 19, 20,  0,  0,  0,  0,  0,  0,  0,  0,
           0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [21, 22, 18, 23,  4, 24, 15, 25, 26, 27, 28, 29, 30, 31,  0,  0,  0,  0,
           0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [32, 33, 18, 34, 35, 36, 21, 22, 18, 37, 38,  0,  0,  0,  0,  0,  0,  0,
           0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [39, 40, 41, 10, 42, 43, 44, 45, 46, 47, 48, 18, 49,  0,  0,  0,  0,  0,
           0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [50, 18, 33, 30, 51, 52, 53, 54, 47, 53, 55,  2, 56, 57, 58, 59, 60,  0,
           0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [61, 18, 62, 63,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
           0,  0,  0,  0,  0,  0, 

In [22]:
# experimental
val_present_ids1 = (tensor_ids1 != 0)
val_present_ids1.shape, type(val_present_ids1), val_present_ids1

(torch.Size([10, 30]),
 torch.Tensor,
 tensor([[ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
           True,  True,  True,  True, False, False, False, False, False, False,
          False, False, False, False, False, False, False, False, False, False],
         [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          False, False, False, False, False, False, False, False, False, False,
          False, False, False, False, False, False, False, False, False, False],
         [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
           True,  True,  True,  True, False, False, False, False, False, False,
          False, False, False, False, False, False, False, False, False, False],
         [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
           True, False, False, False, False, False, False, False, False, False,
          False, False, False, False, False, False, False, False, False, False]

In [23]:
# experimental
tensor_dim_inc_ids1 = val_present_ids1.unsqueeze(-1)
tensor_dim_inc_ids1.shape, type(tensor_dim_inc_ids1), tensor_dim_inc_ids1

(torch.Size([10, 30, 1]),
 torch.Tensor,
 tensor([[[ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False]],
 
         [[ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [ True],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
          [False],
      

In [24]:
# experimental
mask_ids1 = tensor_dim_inc_ids1.float()
mask_ids1.shape, type(mask_ids1), mask_ids1

(torch.Size([10, 30, 1]),
 torch.Tensor,
 tensor([[[1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.]],
 
         [[1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0

In [25]:
# experimental
embedding_matrix = embedding_matrix.to('cuda')
embedding_matrix.shape, type(embedding_matrix), embedding_matrix

(torch.Size([28549, 128]),
 torch.Tensor,
 tensor([[ 0.3756,  0.2267, -0.0065,  ..., -0.2938,  0.4096,  1.8223],
         [ 0.0036,  0.0667,  0.4862,  ..., -0.1020,  0.0392, -0.2243],
         [ 0.2502,  0.0046,  1.3010,  ...,  0.1616, -0.0993, -0.0726],
         ...,
         [ 0.9871,  0.8178, -0.6882,  ..., -1.6554, -2.7404, -0.2003],
         [ 0.7834,  2.6968,  2.5663,  ..., -1.0214,  2.3385,  0.3786],
         [-1.4707,  0.7946,  0.0672,  ..., -0.9628,  1.7868, -1.3583]],
        device='cuda:0', dtype=torch.float64))

In [26]:
# experimental
tensor_ids1.shape, type(tensor_ids1), tensor_ids1

(torch.Size([10, 30]),
 torch.Tensor,
 tensor([[ 2,  3,  4,  5,  6,  5,  7,  8,  9, 10, 11, 12, 10, 13,  0,  0,  0,  0,
           0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [ 2,  3,  4, 14, 15, 16, 17, 18, 19, 20,  0,  0,  0,  0,  0,  0,  0,  0,
           0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [21, 22, 18, 23,  4, 24, 15, 25, 26, 27, 28, 29, 30, 31,  0,  0,  0,  0,
           0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [32, 33, 18, 34, 35, 36, 21, 22, 18, 37, 38,  0,  0,  0,  0,  0,  0,  0,
           0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [39, 40, 41, 10, 42, 43, 44, 45, 46, 47, 48, 18, 49,  0,  0,  0,  0,  0,
           0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [50, 18, 33, 30, 51, 52, 53, 54, 47, 53, 55,  2, 56, 57, 58, 59, 60,  0,
           0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
         [61, 18, 62, 63,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
           0,  0,  0,  0,  0,  0, 

In [27]:
# experimental
sent_embeddings = embedding_matrix[tensor_ids1]
sent_embeddings.shape, type(sent_embeddings), sent_embeddings

(torch.Size([10, 30, 128]),
 torch.Tensor,
 tensor([[[ 0.2502,  0.0046,  1.3010,  ...,  0.1616, -0.0993, -0.0726],
          [ 0.2725, -0.0241,  0.7689,  ..., -0.0866, -0.1411, -0.1148],
          [ 0.2031, -0.0601,  0.6642,  ...,  0.0966, -0.0359, -0.0737],
          ...,
          [ 0.3756,  0.2267, -0.0065,  ..., -0.2938,  0.4096,  1.8223],
          [ 0.3756,  0.2267, -0.0065,  ..., -0.2938,  0.4096,  1.8223],
          [ 0.3756,  0.2267, -0.0065,  ..., -0.2938,  0.4096,  1.8223]],
 
         [[ 0.2502,  0.0046,  1.3010,  ...,  0.1616, -0.0993, -0.0726],
          [ 0.2725, -0.0241,  0.7689,  ..., -0.0866, -0.1411, -0.1148],
          [ 0.2031, -0.0601,  0.6642,  ...,  0.0966, -0.0359, -0.0737],
          ...,
          [ 0.3756,  0.2267, -0.0065,  ..., -0.2938,  0.4096,  1.8223],
          [ 0.3756,  0.2267, -0.0065,  ..., -0.2938,  0.4096,  1.8223],
          [ 0.3756,  0.2267, -0.0065,  ..., -0.2938,  0.4096,  1.8223]],
 
         [[-0.2270,  0.1283,  1.1229,  ..., -0.0582,  0.1

In [28]:
# experimental
mask_ids1.shape, type(mask_ids1), mask_ids1

(torch.Size([10, 30, 1]),
 torch.Tensor,
 tensor([[[1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.]],
 
         [[1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [1.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0.],
          [0

In [29]:
# experimental
#sent_embeddings = sent_embeddings.to('cpu')
#mask_ids1 = mask_ids1.to('cpu')
sent_embeddings = sent_embeddings * mask_ids1
sent_embeddings.shape, type(sent_embeddings), sent_embeddings

(torch.Size([10, 30, 128]),
 torch.Tensor,
 tensor([[[ 0.2502,  0.0046,  1.3010,  ...,  0.1616, -0.0993, -0.0726],
          [ 0.2725, -0.0241,  0.7689,  ..., -0.0866, -0.1411, -0.1148],
          [ 0.2031, -0.0601,  0.6642,  ...,  0.0966, -0.0359, -0.0737],
          ...,
          [ 0.0000,  0.0000, -0.0000,  ..., -0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000, -0.0000,  ..., -0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000, -0.0000,  ..., -0.0000,  0.0000,  0.0000]],
 
         [[ 0.2502,  0.0046,  1.3010,  ...,  0.1616, -0.0993, -0.0726],
          [ 0.2725, -0.0241,  0.7689,  ..., -0.0866, -0.1411, -0.1148],
          [ 0.2031, -0.0601,  0.6642,  ...,  0.0966, -0.0359, -0.0737],
          ...,
          [ 0.0000,  0.0000, -0.0000,  ..., -0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000, -0.0000,  ..., -0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000, -0.0000,  ..., -0.0000,  0.0000,  0.0000]],
 
         [[-0.2270,  0.1283,  1.1229,  ..., -0.0582,  0.1

In [30]:
# experimental
summed = sent_embeddings.sum(dim=1)
summed.shape, type(summed), summed

(torch.Size([10, 128]),
 torch.Tensor,
 tensor([[ 5.5996, -0.4232,  6.6971,  ...,  0.7265,  0.8078,  0.8118],
         [ 2.2912,  1.6873,  5.0731,  ...,  0.2615, -1.7933, -2.0209],
         [-0.1489, -0.1420,  7.4448,  ...,  0.0884,  1.1871, -0.0134],
         ...,
         [-0.7114,  1.6914,  4.9712,  ..., -0.1387, -1.3843,  1.6172],
         [-1.1903,  0.8533,  3.9969,  ..., -0.5568,  0.5629, -0.3409],
         [ 0.1603,  3.7956,  5.4616,  ...,  1.2655, -0.2438,  3.7150]],
        device='cuda:0', dtype=torch.float64))

In [32]:
# experimental
counts = mask_ids1.sum(dim=1).clamp(min=1)
counts.shape, type(counts), counts

(torch.Size([10, 1]),
 torch.Tensor,
 tensor([[14.],
         [10.],
         [14.],
         [11.],
         [13.],
         [17.],
         [ 4.],
         [ 7.],
         [ 8.],
         [ 9.]], device='cuda:0'))

In [33]:
# experimental
# finding sentence mean embedding using torch
sent_final_emb = summed/counts
sent_final_emb.shape, type(sent_final_emb), sent_final_emb

(torch.Size([10, 128]),
 torch.Tensor,
 tensor([[ 0.4000, -0.0302,  0.4784,  ...,  0.0519,  0.0577,  0.0580],
         [ 0.2291,  0.1687,  0.5073,  ...,  0.0262, -0.1793, -0.2021],
         [-0.0106, -0.0101,  0.5318,  ...,  0.0063,  0.0848, -0.0010],
         ...,
         [-0.1016,  0.2416,  0.7102,  ..., -0.0198, -0.1978,  0.2310],
         [-0.1488,  0.1067,  0.4996,  ..., -0.0696,  0.0704, -0.0426],
         [ 0.0178,  0.4217,  0.6068,  ...,  0.1406, -0.0271,  0.4128]],
        device='cuda:0', dtype=torch.float64))

In [42]:
# experimental
# finding sentence mean embedding using numpy
avged = sent_embeddings.to('cpu')
avged = avged.numpy()
avged = np.mean(avged, axis=1, where=(avged!=0))
avged.shape, type(avged), avged

((10, 128),
 numpy.ndarray,
 array([[ 0.39997204, -0.03022999,  0.47836254, ...,  0.05189177,
          0.05769671,  0.05798793],
        [ 0.22912317,  0.16872509,  0.50730937, ...,  0.02615206,
         -0.17933192, -0.20208742],
        [-0.01063746, -0.01014041,  0.53177277, ...,  0.00631134,
          0.08479104, -0.00095838],
        ...,
        [-0.10163006,  0.24163021,  0.71017342, ..., -0.01981282,
         -0.19775655,  0.23102853],
        [-0.14878262,  0.10666122,  0.49961542, ..., -0.06960102,
          0.07036405, -0.04261337],
        [ 0.01781569,  0.42172958,  0.60684918, ...,  0.14061396,
         -0.02708994,  0.41278171]], shape=(10, 128)))

In [43]:
# Create Sentence Embeddings
def ids_to_embedding(id_list_batch): #input is np.array of shape batch_size x 30
    # covert np.array to torch.tensor, dimension = (rows, words per row (=30))
    ids_tensor = torch.tensor(id_list_batch, dtype=torch.long, device=DEVICE)
    
    mask = (ids_tensor != 0)    #Convert tensor to True if there is a value, else False
    mask = mask.unsqueeze(-1)   #Increase a dimension at the end
    mask = mask.float()         #Convert True to 1, False to 0
    
    vecs = embedding_matrix[ids_tensor] # rows, words, 128 per word
    vecs = vecs * mask #zero embeddings for padded word
    
    summed = vecs.sum(dim=1)    # add all word vectors for each row (rows,128)
    counts = mask.sum(dim=1).clamp(min=1)    #count all words per row (rows, 1)
    return summed / counts     #return mean of non-zeros only (rows, 128)

In [46]:
BATCH = 8192
q1_embs, q2_embs = [], []
q1_arr = np.stack(df_train['ids1'].values)
q2_arr = np.stack(df_train['ids2'].values)
for i in range(0, len(df_train), BATCH):
    q1_embs.append(ids_to_embedding(q1_arr[i:i+BATCH]).cpu().numpy())
    q2_embs.append(ids_to_embedding(q2_arr[i:i+BATCH]).cpu().numpy())
q1_emb = np.concatenate(q1_embs)
q2_emb = np.concatenate(q2_embs)

In [51]:
np.savetxt('datasets/Quora Questions Pair Dataset/08_q1_emb.csv', q1_emb, delimiter=",", fmt="%f")
np.savetxt('datasets/Quora Questions Pair Dataset/09_q2_emb.csv', q2_emb, delimiter=",", fmt="%f")

((403968, 128),
 numpy.ndarray,
 array([[ 0.39997204, -0.03022999,  0.47836254, ...,  0.05189177,
          0.05769671,  0.05798793],
        [ 0.22912317,  0.16872509,  0.50730937, ...,  0.02615206,
         -0.17933192, -0.20208742],
        [-0.01063746, -0.01014041,  0.53177277, ...,  0.00631134,
          0.08479104, -0.00095838],
        ...,
        [ 0.0867412 ,  0.05583438,  0.49116638, ...,  0.0435457 ,
          0.02332334, -0.15059494],
        [ 0.2240933 ,  0.01481043,  0.56834665, ..., -0.0930071 ,
          0.1102096 , -0.04013458],
        [ 0.10065736,  0.0242947 ,  0.69339567, ..., -0.14060204,
          0.08488111, -0.13059224]], shape=(403968, 128)))

In [17]:
#load q1 and q2 embeddings from numpy arrays
q1_emb = np.loadtxt('datasets/Quora Questions Pair Dataset/08_q1_emb.csv', delimiter=",")
q2_emb = np.loadtxt('datasets/Quora Questions Pair Dataset/09_q2_emb.csv', delimiter=",")

In [18]:
q1_emb

array([[ 0.399972, -0.03023 ,  0.478363, ...,  0.051892,  0.057697,
         0.057988],
       [ 0.229123,  0.168725,  0.507309, ...,  0.026152, -0.179332,
        -0.202087],
       [-0.010637, -0.01014 ,  0.531773, ...,  0.006311,  0.084791,
        -0.000958],
       ...,
       [ 0.086741,  0.055834,  0.491166, ...,  0.043546,  0.023323,
        -0.150595],
       [ 0.224093,  0.01481 ,  0.568347, ..., -0.093007,  0.11021 ,
        -0.040135],
       [ 0.100657,  0.024295,  0.693396, ..., -0.140602,  0.084881,
        -0.130592]], shape=(403968, 128))